In [ ]:
# Cell 0 — VRAM Check (run first, must show ~7.5GB free)
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"VRAM free  : {free/1024**3:.2f} GB")
print(f"VRAM total : {total/1024**3:.2f} GB")
print(f"VRAM used  : {(total-free)/1024**3:.2f} GB")

if free < 5 * 1024**3:
    print("❌ Too much VRAM in use — kill all python.exe in Task Manager and restart")
else:
    print("✅ VRAM clear — safe to proceed")

In [ ]:
# Cell 1 — Install Dependencies
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip uninstall trl -y -q
!pip install trl==0.8.6 transformers peft bitsandbytes accelerate datasets -q

In [ ]:
# Cell 2 — Imports
import json
import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer

print("All imports successful")

In [ ]:
# Cell 3 — Verify GPU
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU name       : {torch.cuda.get_device_name(0)}")
print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Cell 4 — Define All Paths
import os

MODEL_PATH     = os.path.join(os.path.expanduser("~"), "Desktop", "Qwen_CustomDomain", "Model", "Qwen2.5-3B-Instruct")
DATA_PATH      = os.path.join(os.path.expanduser("~"), "Desktop", "Qwen_CustomDomain", "Dataset", "qa_pairs_100.jsonl")
CHECKPOINT_DIR = os.path.join(os.path.expanduser("~"), "Desktop", "Qwen_CustomDomain", "Checkpoints")
ADAPTER_PATH   = os.path.join(os.path.expanduser("~"), "Desktop", "Qwen_CustomDomain", "Adapters")
MERGED_PATH    = os.path.join(os.path.expanduser("~"), "Desktop", "Qwen_CustomDomain", "MergedModel")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_PATH,   exist_ok=True)
os.makedirs(MERGED_PATH,    exist_ok=True)

print(f"Model path : {MODEL_PATH}")
print(f"Exists     : {os.path.exists(MODEL_PATH)}")
print("All paths ready")

In [ ]:
# Cell 5 — Load and Format Dataset
data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

print(f"Total records loaded : {len(data)}")

def format_record(item):
    system = item["messages"][0]["content"]
    user   = item["messages"][1]["content"]
    asst   = item["messages"][2]["content"]
    text = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{asst}<|im_end|>"
    )
    return {"text": text}

formatted  = [format_record(d) for d in data]
dataset    = Dataset.from_list(formatted)
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Train records : {len(train_data)}")
print(f"Val records   : {len(val_data)}")

In [ ]:
# Cell 6 — 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # ← bf16, not fp16 (Blackwell)
    bnb_4bit_use_double_quant=True
)

print("BitsAndBytes config ready")

In [ ]:
# Cell 7 — Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded")
print(f"Vocab size : {tokenizer.vocab_size}")

In [ ]:
# Cell 8 — Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

print("Base model loaded")
print(f"VRAM used : {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

In [ ]:
# Cell 9 — LoRA Config (8GB safe)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],    # ← only 2 modules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 10 — Training Arguments (8GB safe)
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,         # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    optim="adafactor",
    logging_steps=50,
    eval_strategy="epoch",                 # ← eval after every epoch
    save_strategy="epoch",                 # ← save after every epoch
    save_total_limit=4,                    # ← keep epoch1, epoch2, epoch3 + best
    load_best_model_at_end=True,           # ← auto-saves best val loss model
    metric_for_best_model="eval_loss",     # ← track val loss for best model
    greater_is_better=False,               # ← lower loss = better
    dataloader_num_workers=0,
    report_to="none",
    group_by_length=True,
    gradient_checkpointing=False,
    torch_empty_cache_steps=1,
)
print("Training arguments ready")

In [ ]:
# Cell 11 — Initialize Trainer (Fixed — uses Trainer directly, no SFTTrainer quirks)
import math
from transformers import Trainer, DataCollatorForLanguageModeling

# Pre-tokenize dataset
def tokenize(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        max_length=256,
        padding=False,
    )

train_tokenized = train_data.map(tokenize, remove_columns=["text"])
val_tokenized   = val_data.map(tokenize,   remove_columns=["text"])

# Required: set format for PyTorch tensors
train_tokenized.set_format("torch")
val_tokenized.set_format("torch")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

trainer = Trainer(                         # ← plain Trainer, no SFTTrainer
    model=model,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    args=training_args,
    data_collator=data_collator,
)

# Verify step count
steps_per_epoch = math.ceil(
    len(train_tokenized) /
    (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
)
print(f"Trainer initialized")
print(f"Train samples     : {len(train_tokenized)}")
print(f"Steps per epoch   : {steps_per_epoch}")
print(f"Total steps       : {steps_per_epoch * int(training_args.num_train_epochs)}")

In [ ]:
# Cell 12 — Start Training (Optimized)
import math
import random
import torch
import json
import functools
from transformers import TrainerCallback

# ── Fix PyTorch 2.9 use_reentrant warning ─────────────────
torch.utils.checkpoint.checkpoint = functools.partial(
    torch.utils.checkpoint.checkpoint, use_reentrant=False
)

# ── Live Logging Callback ──────────────────────────────────
class LiveLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step        = state.global_step
        total_steps = state.max_steps
        epoch       = state.epoch or 0

        train_loss = logs.get("loss", None)
        val_loss   = logs.get("eval_loss", None)
        lr         = logs.get("learning_rate", None)

        parts = [f"Step {step:>4}/{total_steps} | Epoch {epoch:.2f}"]
        if train_loss is not None:
            parts.append(f"Train Loss: {train_loss:.4f}")
        if val_loss is not None:
            parts.append(f"Val Loss: {val_loss:.4f}")
        if lr is not None:
            parts.append(f"LR: {lr:.2e}")

        print(" | ".join(parts))

# ── Training ──────────────────────────────────────────────
print("Starting fine-tuning from scratch...")
print(f"{'Step':>9} | {'Epoch':<11} | {'Train Loss':<12} | {'Val Loss':<10} | {'LR'}")
print("-" * 65)

trainer.add_callback(LiveLogCallback())
train_result = trainer.train()
print("\nTraining complete ✅")

# Save adapters immediately
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapters saved to : {ADAPTER_PATH}")

# ── Training vs Validation Loss Gap ───────────────────────
print("\n" + "="*55)
print("   LOSS REPORT")
print("="*55)

train_loss = train_result.training_loss
print(f"Final Training Loss   : {train_loss:.4f}")

eval_result = trainer.evaluate()
val_loss    = eval_result["eval_loss"]
loss_gap    = abs(train_loss - val_loss)

print(f"Final Validation Loss : {val_loss:.4f}")
print(f"Loss Gap              : {loss_gap:.4f}")

if loss_gap < 0.3:
    print("Gap Status            : ✅ Healthy — no overfitting")
elif loss_gap < 0.6:
    print("Gap Status            : ⚠️  Slight overfit — acceptable")
else:
    print("Gap Status            : ❌ Overfitting — reduce epochs or add dropout")

# ── Perplexity ────────────────────────────────────────────
print("\n" + "="*55)
print("   PERPLEXITY REPORT")
print("="*55)

model.eval()

def compute_perplexity(texts, model, tokenizer, max_length=256):
    total_loss   = 0
    total_tokens = 0
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to(model.device)
        with torch.no_grad():
            outputs      = model(**inputs, labels=inputs["input_ids"])
            num_tokens   = inputs["input_ids"].shape[1]
            total_loss  += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    return math.exp(total_loss / total_tokens)

sample_texts = [item["text"] for item in val_data.select(range(min(100, len(val_data))))]
ppl = compute_perplexity(sample_texts, model, tokenizer)

print(f"Perplexity            : {ppl:.2f}")
print(f"Target                : < 10")
if ppl < 10:
    print(f"Status                : ✅ Target achieved")
elif ppl < 15:
    print(f"Status                : ⚠️  Close — run 1 more epoch")
else:
    print(f"Status                : ❌ Run 2 more epochs or increase LoRA rank")

# ── Live Model Output Test ────────────────────────────────
print("\n" + "="*55)
print("   LIVE MODEL OUTPUT TEST")
print("="*55)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    all_data = [json.loads(line) for line in f if line.strip()]

n_samples = min(5, len(all_data))
samples   = random.sample(all_data, n_samples)

def ask_model(question, model, tokenizer):
    prompt = (
        f"<|im_start|>system\n"
        f"You are a Java expert assistant. Answer only from your Java training knowledge. "
        f"If the question is not related to Java say exactly: "
        f"I don't have enough information in my knowledge base to answer this.<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    response   = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

for i, sample in enumerate(samples):
    question     = sample["messages"][1]["content"].split("Question:")[-1].strip()
    expected_ans = sample["messages"][2]["content"]
    model_ans    = ask_model(question, model, tokenizer)

    expected_key = expected_ans[:80].lower()
    model_lower  = model_ans.lower()
    match_score  = sum(1 for word in expected_key.split() if word in model_lower)
    match_ratio  = match_score / max(len(expected_key.split()), 1)

    if match_ratio > 0.6:
        match_label = "✅ Good match"
    elif match_ratio > 0.3:
        match_label = "🔍 Partial match"
    else:
        match_label = "❌ Different — check training"

    print(f"\n[{i+1}] Question : {question}")
    print(f"     Expected : {expected_ans[:200]}")
    print(f"     Model    : {model_ans[:200]}")
    print(f"     Match    : {match_label}  ({match_ratio*100:.0f}% keyword overlap)")
    print("-" * 55)

print("\n✅ Full evaluation complete")

In [ ]:
# Cell 13 — Resume Training (PyTorch 2.6 Compatible)

import os

# Find latest checkpoint
checkpoints = [
    os.path.join(CHECKPOINT_DIR, d)
    for d in os.listdir(CHECKPOINT_DIR)
    if d.startswith("checkpoint-")
]

if not checkpoints:
    print("No checkpoint found. Run Cell 12 to start fresh.")

else:
    latest_checkpoint = max(checkpoints, key=os.path.getmtime)

    print("=" * 60)
    print(f"Resuming from : {latest_checkpoint}")
    print("=" * 60)

    # Remove problematic RNG state file
    rng_file = os.path.join(latest_checkpoint, "rng_state.pth")

    if os.path.exists(rng_file):
        os.remove(rng_file)
        print("Removed rng_state.pth (PyTorch 2.6 compatibility fix)")

    # Resume training
    trainer.train(resume_from_checkpoint=latest_checkpoint)

    print("\nTraining complete")

    # Save final adapters
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)

    print(f"Adapters saved to : {ADAPTER_PATH}")

In [ ]:
# Cell 14 — Merge Adapters Into Base Model (8GB GPU Safe)

import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map=None,          # IMPORTANT
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("Merging adapter weights...")
merged_model = model.merge_and_unload()

print("Saving merged model...")

merged_model.save_pretrained(
    MERGED_PATH,
    safe_serialization=True,
    max_shard_size="2GB"
)

tokenizer.save_pretrained(MERGED_PATH)

print(f"✅ Merged model saved to:\n{MERGED_PATH}")

In [ ]:
# Cell 15 — Test the Fine-Tuned Model
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MERGED_PATH,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,            # ← was float16
    device_map="auto"
)

def ask(question):
    prompt = (
        f"<|im_start|>system\n"
        f"You are a Java expert assistant. Answer only from your Java training knowledge. "
        f"If the question is not related to Java say exactly: "
        f"I don't have enough information in my knowledge base to answer this.<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    out = pipe(prompt, max_new_tokens=256, do_sample=False)
    response = out[0]["generated_text"].split("<|im_start|>assistant\n")[-1]
    print(f"Q : {question}")
    print(f"A : {response}")
    print("-" * 60)

# In-domain Java questions
ask("What is polymorphism in Java?")
ask("How does ArrayList differ from LinkedList?")
ask("What is the use of the finally block in Java?")

# Out-of-scope — should trigger refusal
ask("What is the capital of France?")
ask("How do I train a neural network in Python?")
ask("What is photosynthesis?")